# 01 — RibFrac Patient-Level Data Loading

## Purpose
<p>This notebook establishes the patient-level data loading pipeline for the
RibFrac chest CT dataset used in the development of the proposed
Enhanced 3D ResUNet-based framework for automatic detection,
segmentation, and classification of rib fractures.</p>

<h4>The notebook performs:</h4>
<ol>
<li> Patient-level dataset management</li>
<li> Automatic retrieval of CT images </li>
<li> Automatic retrieval of corresponding annotations </li>
<li> Association of annotation labels with fracture metadata</li>
<li> CT–annotation spatial validation</li>
<li> Memory-efficient remote access to the RibFrac CT archive </li>
<li> Preparation of validated patient data for downstream preprocessing</li>
<li> Patient-level dataset partitioning to prevent data leakage</li>
</ol>

<p>The CT archive is accessed through HTTP range requests. Individual
patients are retrieved without downloading the complete RibFrac CT archive.</p>

## 1. Import Libraries

In [ ]:
#1. Import libraries
import os
import io
import struct
import zipfile
import zlib
import requests

import numpy as np
import pandas as pd
import nibabel as nib

from pathlib import Path
from collections import defaultdict
from sklearn.model_selection import train_test_split

## 2. Define Project Directories

In [ ]:
#2. Define project directories
BASE_DIR = Path("/kaggle/working")

CT_DIR = BASE_DIR / "ribfrac_images"
LABEL_DIR = BASE_DIR / "ribfrac_labels"
METADATA_DIR = BASE_DIR / "ribfrac_metadata"

for directory in [CT_DIR, LABEL_DIR, METADATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project directories:")
print("CT:", CT_DIR)
print("Labels:", LABEL_DIR)
print("Metadata:", METADATA_DIR)

## 3. Define RibFrac Source Information

In [ ]:
#3. Define RibFrac source information

ZENODO_RECORD_PART1 = "https://zenodo.org/records/3893508"

CT_ZIP_URL = ( "https://zenodo.org/records/3893508/files/"
               "ribfrac-train-images-1.zip?download=1" )

LABEL_ZIP_URL = ( "https://zenodo.org/records/3893508/files/"
                  "ribfrac-train-labels-1.zip?download=1" )

INFO_URL = ( "https://zenodo.org/records/3893508/files/"
             "ribfrac-train-info-1.csv?download=1" )

print("RibFrac Part 1 source configured.")

## 4. Download metadata

In [ ]:
#4. Download metadata
INFO_PATH = Path("/kaggle/working/ribfrac-train-info-1.csv")

API_URL = "https://zenodo.org/api/records/3893508"

print("Querying Zenodo record...")

response = requests.get( API_URL, timeout=120 )

print("HTTP status:", response.status_code)

response.raise_for_status()

record = response.json()

print("Zenodo record retrieved.")
print("Title:", record["metadata"]["title"])

print("\nFiles available in the record:")

for file_info in record["files"]:
    print( file_info["key"], "->",  file_info.get("links", {}).get("self") )

In [ ]:
INFO_PATH = "/kaggle/working/ribfrac_metadata/ribfrac-train-info-1.csv"

info_df = pd.read_csv(INFO_PATH)

print("Metadata loaded successfully.")
print("Shape:", info_df.shape)

print("\nColumns:")
print(info_df.columns.tolist())

print("\nFirst 10 rows:")
display(info_df.head(10))

## 5. Load Metadata

In [ ]:
#5. Load metadata
info_df = pd.read_csv(INFO_PATH)

print("Metadata shape:", info_df.shape)
print("\nColumns:")
print(info_df.columns.tolist())

print("\nFirst 10 rows:")
display(info_df.head(10))

## 6. Examine patients

In [ ]:
#6. Examine patients
patient_ids = sorted(info_df["public_id"].unique())

print("Number of unique patients:", len(patient_ids))
print("\nFirst 20 patients:")
print(patient_ids[:20])

## 7. Build patient-level annotation table

In [ ]:
#7. Build patient-level annotation table
patient_annotations = (
    info_df
    .groupby("public_id")
    .agg(
        number_of_annotations=("label_id", "count"),
        label_codes=("label_code", lambda x: sorted(x.dropna().unique().tolist()))
    )
    .reset_index()
)

print("Patient-level annotation table:")
display(patient_annotations.head(10))

## 8. Count fracture codes

In [ ]:
#8. Count fracture codes
print("Label-code distribution:")
print(info_df["label_code"].value_counts(dropna=False).sort_index())

## 9. Download the label archive if necessary

In [ ]:
#9. Download the label archive if necessary
LABEL_ZIP_PATH = BASE_DIR / "ribfrac-train-labels-1.zip"

if not LABEL_ZIP_PATH.exists():

    print("Downloading label archive...")

    response = requests.get(
        LABEL_ZIP_URL,
        stream=True,
        timeout=120
    )

    response.raise_for_status()

    with open(LABEL_ZIP_PATH, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    print("Label archive downloaded.")

else:
    print("Label archive already exists.")

print("Label ZIP:")
print(LABEL_ZIP_PATH)

## 10. Build a label-file index

In [ ]:
#10. Build a label-file index
with zipfile.ZipFile(LABEL_ZIP_PATH, "r") as z:

    label_members = [
        name
        for name in z.namelist()
        if name.endswith("-label.nii.gz")
    ]

print("Number of label files:", len(label_members))

print("\nFirst 10 label files:")
for name in label_members[:10]:
    print(name)

## 11. Create patient → label mapping

In [ ]:
#11. Create patient → label mapping
label_index = {}

for member in label_members:

    filename = Path(member).name

    patient_id = filename.replace("-label.nii.gz", "")

    label_index[patient_id] = member

print("Patients indexed:", len(label_index))

print("\nExample:")
print("RibFrac128 →", label_index.get("RibFrac128"))

## 12. Extract a patient's label

In [ ]:
#12. Extract a patient's label
def get_label_path(patient_id):
    """
    Extract and return the correct local path
    to a patient's RibFrac label.
    """

    if patient_id not in label_index:
        raise ValueError(
            f"No label found for patient: {patient_id}"
        )

    member = label_index[patient_id]

    # Preserve the ZIP directory structure
    output_path = LABEL_DIR / member

    # Create required parent directories
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # Extract only if the file does not already exist
    if not output_path.exists():

        with zipfile.ZipFile(LABEL_ZIP_PATH, "r") as z:
            z.extract(member, LABEL_DIR)

    return output_path

In [ ]:
label_path = get_label_path("RibFrac128")
print(label_path)

## 13. Build the remote CT ZIP index

In [ ]:
#13. Build the remote CT ZIP index
def http_range(start, end):
    """
    Download bytes [start, end] from the remote CT ZIP.
    """

    headers = {
        "Range": f"bytes={start}-{end}"
    }

    response = requests.get(
        CT_ZIP_URL,
        headers=headers,
        timeout=120
    )

    response.raise_for_status()

    return response.content

## 14. Read ZIP64 information

In [ ]:
#14 Read ZIP64 information
def find_zip64_eocd():
    """
    Locate the ZIP64 End of Central Directory record.
    """

    # Read the last 64 KB
    tail_size = 65536

    response = requests.get(
        CT_ZIP_URL,
        headers={"Range": f"bytes=-{tail_size}"},
        timeout=120
    )

    response.raise_for_status()

    data = response.content

    # ZIP64 locator
    locator_signature = b"PK\x06\x07"

    pos = data.rfind(locator_signature)

    if pos == -1:
        raise RuntimeError(
            "ZIP64 End of Central Directory locator not found."
        )

    # ZIP64 EOCD offset is bytes 8–15 after locator signature
    zip64_eocd_offset = struct.unpack_from(
        "<Q",
        data,
        pos + 8
    )[0]

    return zip64_eocd_offset

## 15. Parse the central directory

In [ ]:
#15. Parse the central directory
def build_ct_zip_index():

    zip64_eocd_offset = find_zip64_eocd()

    print("ZIP64 EOCD offset:", zip64_eocd_offset)

    # Read ZIP64 EOCD
    eocd = http_range(
        zip64_eocd_offset,
        zip64_eocd_offset + 55
    )

    if eocd[:4] != b"PK\x06\x06":
        raise RuntimeError("Invalid ZIP64 EOCD signature.")

    total_entries = struct.unpack_from(
        "<Q", eocd, 32
    )[0]

    central_dir_size = struct.unpack_from(
        "<Q", eocd, 40
    )[0]

    central_dir_offset = struct.unpack_from(
        "<Q", eocd, 48
    )[0]

    print("Total entries:", total_entries)
    print("Central directory size:", central_dir_size)
    print("Central directory offset:", central_dir_offset)

    central_data = http_range(
        central_dir_offset,
        central_dir_offset + central_dir_size - 1
    )

    entries = []

    pos = 0

    while pos < len(central_data):

        if central_data[pos:pos + 4] != b"PK\x01\x02":
            break

        compressed_size = struct.unpack_from(
            "<I", central_data, pos + 20
        )[0]

        uncompressed_size = struct.unpack_from(
            "<I", central_data, pos + 24
        )[0]

        filename_length = struct.unpack_from(
            "<H", central_data, pos + 28
        )[0]

        extra_length = struct.unpack_from(
            "<H", central_data, pos + 30
        )[0]

        comment_length = struct.unpack_from(
            "<H", central_data, pos + 32
        )[0]

        local_header_offset = struct.unpack_from(
            "<I", central_data, pos + 42
        )[0]

        filename_start = pos + 46

        filename_end = (
            filename_start + filename_length
        )

        filename = central_data[
            filename_start:filename_end
        ].decode("utf-8")

        entries.append({
            "filename": filename,
            "compressed_size": compressed_size,
            "uncompressed_size": uncompressed_size,
            "local_header_offset": local_header_offset
        })

        pos = (
            filename_end
            + extra_length
            + comment_length
        )

    return pd.DataFrame(entries)

In [ ]:
ct_zip_index = build_ct_zip_index()

print("\nCT ZIP index created.")
print("Entries:", len(ct_zip_index))

display(ct_zip_index.head())

## 16. Correct ZIP64 size handling

In [ ]:
#16. Correct ZIP64 size handling
def get_ct_entry(patient_id):

    filename = f"Part1/{patient_id}-image.nii.gz"

    matches = ct_zip_index[
        ct_zip_index["filename"] == filename
    ]

    if len(matches) == 0:
        raise ValueError(
            f"CT not found in archive: {patient_id}"
        )

    return matches.iloc[0]

In [ ]:
entry = get_ct_entry("RibFrac128")

print(entry)

## 17. Remote CT downloader

In [ ]:
#17. Remote CT downloader
def download_ribfrac_image(
    patient_id,
    chunk_size=1024 * 1024
):

    output_path = CT_DIR / f"{patient_id}-image.nii.gz"

    # Use existing file if available
    if output_path.exists():

        print("CT already exists:")
        print(output_path)

        return output_path

    entry = get_ct_entry(patient_id)

    local_header_offset = int(
        entry["local_header_offset"]
    )

    compressed_size = int(
        entry["compressed_size"]
    )

    uncompressed_size = int(
        entry["uncompressed_size"]
    )

    # Read local ZIP header
    local_header = http_range(
        local_header_offset,
        local_header_offset + 29
    )

    if local_header[:4] != b"PK\x03\x04":
        raise RuntimeError(
            "Invalid local ZIP header."
        )

    filename_length = struct.unpack_from(
        "<H",
        local_header,
        26
    )[0]

    extra_length = struct.unpack_from(
        "<H",
        local_header,
        28
    )[0]

    data_start = (
        local_header_offset
        + 30
        + filename_length
        + extra_length
    )

    data_end = (
        data_start
        + compressed_size
        - 1
    )

    print(f"Patient: {patient_id}")
    print(f"Compressed size: {compressed_size / 1024**2:.2f} MB")
    print(
        f"Uncompressed size: "
        f"{uncompressed_size / 1024**2:.2f} MB"
    )

    # Download compressed member in chunks
    compressed_data = bytearray()

    downloaded = 0

    while downloaded < compressed_size:

        chunk_end = min(
            downloaded + chunk_size,
            compressed_size
        ) - 1

        start = data_start + downloaded
        end = data_start + chunk_end

        chunk = http_range(start, end)

        compressed_data.extend(chunk)

        downloaded += len(chunk)

        print(
            f"\rDownloaded: "
            f"{downloaded / 1024**2:.2f} / "
            f"{compressed_size / 1024**2:.2f} MB",
            end=""
        )

    print("\nDownload complete.")

    # Decompress DEFLATE stream
    print("Decompressing...")

    decompressed = zlib.decompress(
        bytes(compressed_data),
        -15
    )

    print(
        f"Decompressed size: "
        f"{len(decompressed) / 1024**2:.2f} MB"
    )

    # Validate size
    if len(decompressed) != uncompressed_size:

        raise RuntimeError(
            "Decompressed size does not match ZIP metadata."
        )

    # Save .nii.gz
    with open(output_path, "wb") as f:
        f.write(decompressed)

    print("Saved:", output_path)

    return output_path

## 18. Test automatic CT retrieval

In [ ]:
#18. Test automatic CT retrieval
ct_path = download_ribfrac_image("RibFrac128")

print("\nCT path:")
print(ct_path)

## 19. Load a patient

In [ ]:
#19. Load a patient
def load_patient(patient_id):

    print("=" * 60)
    print(f"Loading patient: {patient_id}")
    print("=" * 60)

    # Retrieve CT
    ct_path = download_ribfrac_image(patient_id)

    # Retrieve label
    label_path = get_label_path(patient_id)

    # Load NIfTI
    ct_img = nib.load(ct_path)
    label_img = nib.load(label_path)

    ct_volume = ct_img.get_fdata(dtype=np.float32)
    label_volume = label_img.get_fdata(dtype=np.float32)

    return {
        "patient_id": patient_id,
        "ct_path": ct_path,
        "label_path": label_path,
        "ct_img": ct_img,
        "label_img": label_img,
        "ct_volume": ct_volume,
        "label_volume": label_volume
    }

In [ ]:
patient = load_patient("RibFrac128")

## 20. Patient validation

In [ ]:
#20. Patient validation
def validate_patient(patient):

    ct_img = patient["ct_img"]
    label_img = patient["label_img"]

    ct = patient["ct_volume"]
    label = patient["label_volume"]

    results = {}

    results["ct_is_3d"] = (ct.ndim == 3)

    results["label_is_3d"] = (label.ndim == 3)

    results["shape_match"] = (
        ct_img.shape == label_img.shape
    )

    results["spacing_match"] = np.allclose(
        ct_img.header.get_zooms()[:3],
        label_img.header.get_zooms()[:3]
    )

    results["affine_match"] = np.allclose(
        ct_img.affine,
        label_img.affine
    )

    results["ct_finite"] = np.isfinite(ct).all()

    results["label_finite"] = np.isfinite(label).all()

    results["has_annotation"] = (
        np.count_nonzero(label) > 0
    )

    return results

In [ ]:
validation = validate_patient(patient)

print("PATIENT VALIDATION")
print("=" * 60)

for key, value in validation.items():
    print(f"{key}: {value}")

## 21. Display patient information

In [ ]:
#21. Display patient information

def patient_summary(patient):

    ct_img = patient["ct_img"]
    label = patient["label_volume"]

    print("=" * 60)
    print("PATIENT SUMMARY")
    print("=" * 60)

    print("Patient ID:", patient["patient_id"])

    print("\nCT")
    print("Shape:", ct_img.shape)
    print("Spacing:", ct_img.header.get_zooms()[:3])
    print("Minimum HU:", np.min(patient["ct_volume"]))
    print("Maximum HU:", np.max(patient["ct_volume"]))
    print("Mean HU:", np.mean(patient["ct_volume"]))
    print("Standard deviation:", np.std(patient["ct_volume"]))

    print("\nLABEL")
    print("Shape:", label.shape)
    print("Unique values:", np.unique(label))
    print("Non-zero voxels:", np.count_nonzero(label))

## 22. Associate annotation IDs with fracture

In [ ]:
#22. Associate annotation IDs with fracture
patient_id = "RibFrac128"

patient_info = info_df[
    info_df["public_id"] == patient_id
].copy()

print("Annotation metadata:")
display(patient_info)

## 23. Create a patient annotation dictionary

In [ ]:
#23. Create a patient annotation dictionary
def get_patient_annotation_metadata(patient_id):

    rows = info_df[
        info_df["public_id"] == patient_id
    ].copy()

    annotation_dict = {}

    for _, row in rows.iterrows():

        annotation_id = int(row["label_id"])

        annotation_dict[annotation_id] = {
            "label_code": row["label_code"]
        }

    return annotation_dict

In [ ]:
annotation_metadata = get_patient_annotation_metadata(
    "RibFrac128"
)

annotation_metadata

## 24. Check annotation IDs against metadata

In [ ]:
#24. Check annotation IDs against metadata
label_ids_in_nifti = set(
    np.unique(patient["label_volume"]).astype(int)
)

label_ids_in_nifti.discard(0)

metadata_ids = set(
    annotation_metadata.keys()
)

print("Annotation IDs in NIfTI:")
print(sorted(label_ids_in_nifti))

print("\nAnnotation IDs in metadata:")
print(sorted(metadata_ids))

print("\nNIfTI IDs missing from metadata:")
print(
    sorted(label_ids_in_nifti - metadata_ids)
)

print("\nMetadata IDs missing from NIfTI:")
print(
    sorted(metadata_ids - label_ids_in_nifti)
)

## 25. Patient-level splitting

In [ ]:
#25. Patient-level splitting
patients = np.array(patient_ids)

train_patients, test_patients = train_test_split(
    patients,
    test_size=0.20,

    random_state=42
)

train_patients, val_patients = train_test_split(
    train_patients,
    test_size=0.20,
    random_state=42
)

print("Train patients:", len(train_patients))
print("Validation patients:", len(val_patients))
print("Test patients:", len(test_patients))

## 26. Verify no patient leakage

In [ ]:
#26. Verify no patient leakage
train_set = set(train_patients)
val_set = set(val_patients)
test_set = set(test_patients)

print("Train ∩ Validation:",
      train_set.intersection(val_set))

print("Train ∩ Test:",
      train_set.intersection(test_set))

print("Validation ∩ Test:",
      val_set.intersection(test_set))

## 27. Save patient split

In [ ]:
#27. Save patient split
split_df = pd.DataFrame({
    "patient_id": (
        list(train_patients)
        + list(val_patients)
        + list(test_patients)
    ),
    "split": (
        ["train"] * len(train_patients)
        + ["validation"] * len(val_patients)
        + ["test"] * len(test_patients)
    )
})

split_path = METADATA_DIR / "patient_split.csv"

split_df.to_csv(
    split_path,
    index=False
)

print("Patient split saved:")
print(split_path)

display(split_df.head())

## 28. Create a final patient manifest

In [ ]:
#28. Create a final patient manifest
manifest = pd.DataFrame({
    "patient_id": patient_ids
})

manifest["ct_member"] = manifest["patient_id"].apply(
    lambda x: f"Part1/{x}-image.nii.gz"
)

manifest["label_member"] = manifest["patient_id"].apply(
    lambda x: label_index.get(x)
)

manifest["has_label"] = (
    manifest["label_member"].notna()
)

manifest = manifest.merge(
    split_df,
    on="patient_id",
    how="left"
)

manifest_path = METADATA_DIR / "ribfrac_patient_manifest.csv"

manifest.to_csv(
    manifest_path,
    index=False
)

print("Manifest saved:")
print(manifest_path)

display(manifest.head(10))

## 29. Final integrity check

In [ ]:
#29. Final integrity check
print("FINAL DATASET CHECK")
print("=" * 60)

print("Unique patients:", manifest["patient_id"].nunique())

print(
    "Patients with labels:",
    manifest["has_label"].sum()
)

print("\nSplit distribution:")
print(manifest["split"].value_counts())

print("\nMissing labels:")
print(
    manifest["has_label"].value_counts()
)

## 30. Final test using RibFrac128

In [ ]:
#30. Final test using RibFrac128
patient_id = "RibFrac128"

patient = load_patient(patient_id)

validation = validate_patient(patient)

print("\nFINAL VALIDATION")
print("=" * 60)

for key, value in validation.items():
    print(f"{key}: {value}")